In [ ]:
"""
notes to work on boosting to 73% accuracy

- use the entire dataset (do not split data) for cross validation and report the mean satndard deviation of accuracy
- remove drop duplicates 
- feature selection: use select best on xgboost to find features that are most important to the model and use them instead of all 6
(X1-X6)
- try logistic regression as baseline instead of decision tree



optional:
build an algorithm that can bactrack the features to show what impact they have
"""
    

In [1]:
from models.CoreModel import CoreModel

In [ ]:
model = CoreModel()
model.load_data_from_csv("../data/ACME-HappinessSurvey2020.csv")

#use entire dataset
x = model.get_data()[["X1", "X2", "X3", "X4", "X5", "X6"]]
y = model.get_data()["Y"] 

In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#logistic regression baseline model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

In [64]:
#load decision tree
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=42, max_depth=2, min_samples_split=2, min_samples_leaf=10)

In [172]:
# load the xgboost model with specified hyperparameters found as best results
from xgboost import XGBClassifier
model = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=2)

In [ ]:
#load knn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        weights="uniform"
    ))
])

In [168]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [173]:
from sklearn.model_selection import RepeatedStratifiedKFold


cv = RepeatedStratifiedKFold(   
    n_splits=5,
    n_repeats=10,
    random_state=42
)

In [174]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score


for k in range(1, 7):
    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    X_selected = selector.fit_transform(x, y)


    scores = cross_val_score(
        model,
        X_selected,
        y,
        cv=cv,
        scoring="accuracy"
    )

    selected_features = x.columns[selector.get_support()]
    print("k: ", k)
    print("Features: ", list(selected_features))
    print("Mean Accuracy: ", scores.mean())
    print("Standard Deviation: ", scores.std())


k:  1
Features:  ['X1']
Mean Accuracy:  0.6508923076923077
Standard Deviation:  0.07398502327254518
k:  2
Features:  ['X1', 'X5']
Mean Accuracy:  0.6476
Standard Deviation:  0.08529151403782152
k:  3
Features:  ['X1', 'X5', 'X6']
Mean Accuracy:  0.6444307692307691
Standard Deviation:  0.08214672397822709
k:  4
Features:  ['X1', 'X3', 'X5', 'X6']
Mean Accuracy:  0.6063999999999999
Standard Deviation:  0.0943629452899123
k:  5
Features:  ['X1', 'X3', 'X4', 'X5', 'X6']
Mean Accuracy:  0.5841846153846153
Standard Deviation:  0.08198371320432632
k:  6
Features:  ['X1', 'X2', 'X3', 'X4', 'X5', 'X6']
Mean Accuracy:  0.5620307692307693
Standard Deviation:  0.0826618920807337


In [121]:
"""Best Results

logistic regresison

k:  1
Features:  ['X1']
Mean Accuracy:  64.25%
Standard Deviation:  0.06372722937107504


Decision Tree

k:  2
Features:  ['X1', 'X5']
Mean Accuracy:  65.82%
Standard Deviation:  0.08413232744141848


XGBoost 

k:  3
Features:  ['X1', 'X5', 'X6']
Mean Accuracy:  68.25%
Standard Deviation:  0.025772789650919278

knn

k:  3
Features:  ['X1', 'X5', 'X6']
Mean Accuracy:  62.65%
Standard Deviation:  0.056955500121435486

"""

"Best Results\n\nlogistic regresison\n\nk:  1\nFeatures:  ['X1']\nMean Accuracy:  64.25%\nStandard Deviation:  0.06372722937107504\n\n\nDecision Tree\n\nk:  2\nFeatures:  ['X1', 'X5']\nMean Accuracy:  65.82%\nStandard Deviation:  0.08413232744141848\n\n\nXGBoost \n\nk:  3\nFeatures:  ['X1', 'X5', 'X6']\nMean Accuracy:  68.25%\nStandard Deviation:  0.025772789650919278\n\nknn\n\nk:  3\nFeatures:  ['X1', 'X5', 'X6']\nMean Accuracy:  62.65%\nStandard Deviation:  0.056955500121435486\n\n"

In [ ]:
"""XGBoost Interesting results

best result with

learning rate = 0.1
max_depth = 2
cv = StratifiedKFold

k:  3
Features:  ['X1', 'X5', 'X6']
Mean Accuracy:  68.25%
Standard Deviation:  0.025772789650919278


second best
learning rate = 0.1
max_depth = 4
cv = RepeatedStratifiedKFold

k:  3
Features:  ['X1', 'X5', 'X6']
Mean Accuracy:  67.07%
Standard Deviation:  0.0777158897949912


Results using RepeatedStratifiedKFold, learning rate = 0.1, max_depth = 2

|  k | Selected features      | Mean Accuracy | Std. Dev. |
| -: | ---------------------- | ------------: | --------: |
|  1 | X1                     |        65.09% |     7.40% |
|  2 | X1, X5                 |        64.76% |     8.53% |
|  3 | X1, X5, X6             |        64.44% |     8.21% |
|  4 | X1, X3, X5, X6         |        60.64% |     9.44% |
|  5 | X1, X3, X4, X5, X6     |        58.42% |     8.20% |
|  6 | X1, X2, X3, X4, X5, X6 |        56.20% |     8.27% |



"""